In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
import json, os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Content-V9'
EXPECTED_EXACT = 'a85301b5d8e53e3bd83bdc891f69d988bf2b06cc'
RUNNER_MODULE = 'experiments.run_content_v9_stability'
SOURCE = pathlib.Path('/content/cegwm-content-v9-source')
LOCAL = pathlib.Path('/content/Content-V9-a85301b-local')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Content')
CAPTURE_LIMIT = 4096


In [ ]:
if SOURCE.exists() or LOCAL.exists(): raise FileExistsError('create-only local path')
subprocess.run(['git','clone','--no-single-branch','--branch',BRANCH,REPO_URL,str(SOURCE)],check=True)
def git(*args): return subprocess.run(['git',*args],cwd=SOURCE,check=True,capture_output=True,text=True).stdout.strip()
git('checkout','--detach',EXPECTED_EXACT)
if git('rev-parse','HEAD') != EXPECTED_EXACT or git('status','--porcelain'): raise RuntimeError('checkout identity')
RUN_UTC=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); DRIVE_TARGET=DRIVE_ROOT/f'Content-V9-{EXPECTED_EXACT[:7]}-{RUN_UTC}'
if DRIVE_TARGET.exists(): raise FileExistsError('create-only Drive target')
subprocess.run([sys.executable,'-m','pip','install',str(SOURCE)],check=True)
if git('rev-parse','HEAD') != EXPECTED_EXACT or git('status','--porcelain') or LOCAL.exists() or DRIVE_TARGET.exists(): raise RuntimeError('post-install identity')
from google.colab import userdata
child_env={k:v for k,v in os.environ.items() if not any(x in k.upper() for x in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}; root_key=token=''
try:
    root_key=userdata.get('CEG_WM_ROOT_KEY'); token=userdata.get('HF_TOKEN'); child_env['CEG_WM_ROOT_KEY']=root_key; child_env['HF_TOKEN']=token
    p=subprocess.Popen([sys.executable,'-m',RUNNER_MODULE,'--repo-root',str(SOURCE),'--expected-exact',EXPECTED_EXACT,'--local-work-root',str(LOCAL),'--artifact-sink',str(DRIVE_TARGET)],cwd=SOURCE,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
finally:
    child_env.pop('CEG_WM_ROOT_KEY',None); child_env.pop('HF_TOKEN',None); root_key=token=''
summary=None; summary_count=0
for raw_line in iter(p.stdout.readline,b''):
    if len(raw_line)>CAPTURE_LIMIT: raise RuntimeError('runner line bound')
    line=raw_line.decode('utf-8','strict').strip()
    if line.startswith('CEGWM_SUMMARY '): summary=json.loads(line[14:]); summary_count+=1
rc=p.wait()
if summary_count!=1 or not isinstance(summary,dict) or summary.get('phase')!='terminal' or summary.get('rc')!=rc or not all(isinstance(summary.get(k),int) and not isinstance(summary.get(k),bool) for k in ('committed','fixed_total')): raise RuntimeError('terminal summary contract')


In [ ]:
if rc==0 and summary['committed']==summary['fixed_total']:
    print('CEGWM_CONTENT_V9_ARTIFACT '+json.dumps({'execution_exact':EXPECTED_EXACT,'artifact_result_path':str(DRIVE_TARGET),'completeness':'complete','scientific_status':'not_adjudicated'},sort_keys=True,separators=(',',':')))
elif rc==2:
    print('CEGWM_CONTENT_V9_INCOMPLETE '+json.dumps({'execution_exact':EXPECTED_EXACT,'artifact_result_path':str(DRIVE_TARGET),'completeness':'incomplete','scientific_status':'not_evaluable'},sort_keys=True,separators=(',',':')))
else: raise RuntimeError('runner completion contract')
